# Exponential smoothing: first and second order: lab notebook

**Statistical Modelling and Forecasting, Regression and exponential smoothing**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, Holt
from statsmodels.tsa.arima.model import ARIMA

### 1. Simple exponential smoothing on a level series: Nile river flow (annual)

In [ ]:
# 1. Simple exponential smoothing on a level series: Nile river flow (annual)
nile = sm.datasets.nile.load_pandas().data["volume"].values
ses = SimpleExpSmoothing(nile, initialization_method="estimated").fit()
print("SES: alpha =", round(ses.params["smoothing_level"], 3), "| SSE =", round(ses.sse, 0), "| forecast (flat):", round(float(ses.forecast(3)[0]), 1))

### 2. The recursion by hand and the weights

In [ ]:
# 2. The recursion by hand and the weights
a, l = 0.3, nile[0]
sse = 0
for v in nile[1:]:
    sse += (v - l) ** 2
    l = a * v + (1 - a) * l
chk = SimpleExpSmoothing(nile, initialization_method="known", initial_level=nile[0]).fit(smoothing_level=a, optimized=False)
print("by hand: level", round(l, 2), "SSE", round(sse, 0), "| statsmodels level", round(chk.level[-1], 2))
print("weights on y_t, y_t-1, y_t-2, ...:", np.round([a * (1 - a) ** k for k in range(6)], 3), "| sum of 40 weights:", round(sum(a * (1 - a) ** k for k in range(40)), 4))

### 3. SES is ARIMA(0,1,1) with theta = alpha - 1

In [ ]:
# 3. SES is ARIMA(0,1,1) with theta = alpha - 1
ar = ARIMA(nile, order=(0, 1, 1)).fit()
print("ARIMA(0,1,1): theta =", round(ar.params[0], 3), "-> alpha = 1 + theta =", round(1 + ar.params[0], 3))

### 4. A trending series: log real GDP. SES lags, Holt follows the slope

In [ ]:
# 4. A trending series: log real GDP. SES lags, Holt follows the slope
g = np.log(sm.datasets.macrodata.load_pandas().data["realgdp"].values)
train, test = g[:190], g[190:]
f1 = SimpleExpSmoothing(train, initialization_method="estimated").fit()
f2 = Holt(train, initialization_method="estimated").fit()
f3 = Holt(train, damped_trend=True, initialization_method="estimated").fit()
rmse = lambda p: float(np.sqrt(np.mean((test - p) ** 2)))
print("alpha (SES):", round(f1.params["smoothing_level"], 3), "| alpha, beta (Holt):", round(f2.params["smoothing_level"], 3), round(f2.params["smoothing_trend"], 3))
print("hold-out RMSE: SES", round(rmse(f1.forecast(len(test))), 4), "| Holt", round(rmse(f2.forecast(len(test))), 4), "| damped Holt", round(rmse(f3.forecast(len(test))), 4))
print("(alpha = 1 means each forecast copies the last value; the 2008-09 recession is in the test period, so the flat forecasts win)")
print("slope carried by Holt per quarter:", round(f2.params["initial_trend"], 4), "| last level:", round(f2.level[-1], 3))

### 5. Brown's double smoothing by hand

In [ ]:
# 5. Brown's double smoothing by hand
def brown(y, a):
    s1 = s2 = y[0]
    for v in y[1:]:
        s1 = a * v + (1 - a) * s1
        s2 = a * s1 + (1 - a) * s2
    return 2 * s1 - s2, a / (1 - a) * (s1 - s2)
lv, sl = brown(train, 0.2)
print("Brown (alpha 0.2): level", round(lv, 4), "slope", round(sl, 5), "| forecast 4 ahead:", round(lv + 4 * sl, 4))
ah, bh = 0.2 * (2 - 0.2), 0.2 / (2 - 0.2)
h = Holt(train[1:], initialization_method="known", initial_level=train[0], initial_trend=train[1] - train[0]).fit(smoothing_level=ah, smoothing_trend=bh, optimized=False)
print("Holt with alpha =", round(ah, 3), "beta =", round(bh, 3), "gives almost the same forecast:", round(float(h.forecast(4)[-1]), 4))

### 6. The numerical problem

In [ ]:
# 6. The numerical problem
y = [21, 24, 25, 29, 31, 32]; al, be = 0.3, 0.2
l = y[0]; sse = 0
for v in y[1:]:
    sse += (v - l) ** 2; l = al * v + (1 - al) * l
lh, bh2 = y[0], y[1] - y[0]
for v in y[1:]:
    ln = al * v + (1 - al) * (lh + bh2); bh2 = be * (ln - lh) + (1 - be) * bh2; lh = ln
print("SES level", round(l, 3), "SSE", round(sse, 3), "| Holt level", round(lh, 3), "slope", round(bh2, 3), "forecast h=3:", round(lh + 3 * bh2, 3))

## Expected output

Running all cells above should print:

```
SES: alpha = 0.246 | SSE = 2038674.0 | forecast (flat): 805.3
by hand: level 788.44 SSE 2043114.0 | statsmodels level 788.44
weights on y_t, y_t-1, y_t-2, ...: [0.3   0.21  0.147 0.103 0.072 0.05 ] | sum of 40 weights: 1.0
ARIMA(0,1,1): theta = -0.733 -> alpha = 1 + theta = 0.267
alpha (SES): 1.0 | alpha, beta (Holt): 1.0 0.0
hold-out RMSE: SES 0.0204 | Holt 0.0559 | damped Holt 0.031
(alpha = 1 means each forecast copies the last value; the 2008-09 recession is in the test period, so the flat forecasts win)
slope carried by Holt per quarter: 0.0083 | last level: 9.47
Brown (alpha 0.2): level 9.4703 slope 0.00735 | forecast 4 ahead: 9.4997
Holt with alpha = 0.36 beta = 0.111 gives almost the same forecast: 9.4997
SES level 28.204 SSE 126.021 | Holt level 33.779 slope 2.647 forecast h=3: 41.72
```

## Practice exercises

Try these in the cells below.

1. On "Trend" data, compare SES with Holt. Where is the SES forecast relative to the data?
2. Slide α from 0.1 to 0.9 and watch the SSE curve. Is the minimum at a large or small α for a noisy level series?
3. Switch to "Optimise parameters". Compare the chosen α with ARIMA(0,1,1)'s θ through α = 1 + θ.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
